# YOLO11s-Seg Segmenter: Custom Training & Inference Dashboard
### Training on Custom High-Quality Dataset (`good_ones_dataset.zip`)

This notebook trains a YOLO11s-Seg model on your curated dataset containing 138+ clean architectural plans, generates evaluation curves, and provides an interactive UI to upload and test any floorplan image after training.

## 1. Environment Setup

In [ ]:
# Mount Google Drive if backup is needed
from google.colab import drive
drive.mount('/content/drive')

# Install YOLO, ONNX, and plotting dependencies
!pip install -q ultralytics onnx onnxslim onnxruntime pandas opencv-python pyyaml matplotlib

import os
import sys
import shutil
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. Please switch Colab Runtime to GPU for training.")

## 2. Load and Extract Dataset (`good_ones_dataset.zip`)
This cell searches your Google Drive and local storage for `good_ones_dataset.zip`. If not found, it prompts you to upload the zip file directly from your computer.

In [ ]:
from google.colab import files
import zipfile

zip_path = None
candidates = [
    "/content/good_ones_dataset.zip",
    "/content/drive/MyDrive/BOM_Project/good_ones_dataset.zip",
    "/content/drive/MyDrive/good_ones_dataset.zip"
]

for c in candidates:
    if os.path.exists(c):
        zip_path = c
        break
        
if zip_path is None:
    print("good_ones_dataset.zip was not found in Drive. Please upload the dataset zip directly:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            zip_path = os.path.join("/content", fn)
            break

if zip_path and os.path.exists(zip_path):
    local_dir = "/content/good_ones_dataset"
    if os.path.exists(local_dir): shutil.rmtree(local_dir)
    print(f"Unpacking dataset zip from {zip_path}...")
    shutil.unpack_archive(zip_path, local_dir, "zip")
    print("Extraction complete!")
else:
    print("ERROR: No dataset zip file could be located.")

## 3. Dynamic YAML Configuration Setup
This cell searches the unzipped contents to locate the image folders, and writes the `dataset.yaml` configuration mapping splits to the labels `0: door` and `1: window`.

In [ ]:
import glob

# Find the path containing images/train in the unzipped folder
search_res = glob.glob("/content/good_ones_dataset/**/images/train", recursive=True)
if search_res:
    dataset_root = os.path.dirname(os.path.dirname(search_res[0]))
else:
    dataset_root = "/content/good_ones_dataset"
    
print(f"Detected Dataset Root: {dataset_root}")

# Create dataset.yaml configuration file
dataset_yaml_content = f"""path: {dataset_root}
train: images/train
val: images/val

names:
  0: door
  1: window
"""
yaml_path = os.path.join(dataset_root, "dataset.yaml")
with open(yaml_path, "w") as f:
    f.write(dataset_yaml_content)
print(f"dataset.yaml written to: {yaml_path}")

## 4. Run YOLO11s-Seg Model Training

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s-seg.pt") # Loads baseline pre-trained weights

print("Starting training on custom high-quality dataset...")
try:
    results = model.train(
        data=yaml_path,
        epochs=100,
        imgsz=1024,
        batch=16,
        device=0,
        cache=True,
        amp=True,
        workers=2,
        project="good_ones_project",
        name="yolo11s_seg_good",
        plots=True,
        save=True,
        optimizer="AdamW",
        cos_lr=True
    )
except Exception as e:
    print(f"Training error: {e}. Retrying with batch=8 memory fallback...")
    results = model.train(
        data=yaml_path,
        epochs=100,
        imgsz=1024,
        batch=8,
        device=0,
        cache=True,
        amp=True,
        workers=2,
        project="good_ones_project",
        name="yolo11s_seg_good",
        plots=True,
        save=True,
        optimizer="AdamW",
        cos_lr=True
    )

## 5. View Evaluation Metrics & Training Curves
Plots loss convergence maps, confusion matrices, and precision/recall curves to assess segment performance.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

run_dir = "good_ones_project/yolo11s_seg_good"
csv_path = os.path.join(run_dir, "results.csv")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    fig, axs = plt.subplots(1, 2, figsize=(16, 5))
    
    axs[0].plot(df['epoch'], df['train/box_loss'], label='train/box_loss')
    axs[0].plot(df['epoch'], df['val/box_loss'], label='val/box_loss')
    axs[0].set_title('Loss Convergence')
    axs[0].set_xlabel('Epoch')
    axs[0].set_ylabel('Loss')
    axs[0].legend()
    
    axs[1].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50 Box')
    axs[1].plot(df['epoch'], df['metrics/mAP50(M)'], label='mAP50 Mask')
    axs[1].set_title('Validation Mean Average Precision')
    axs[1].set_xlabel('Epoch')
    axs[1].set_ylabel('mAP')
    axs[1].legend()
    plt.show()
else:
    print("results.csv not found.")

# Display results logs
for metric_img in ["confusion_matrix.png", "PR_curve.png", "F1_curve.png"]:
    p = os.path.join(run_dir, metric_img)
    if os.path.exists(p):
        print(f"\n=== {metric_img} ===")
        display(Image(filename=p))

## 6. Detailed Model Validation Report

In [ ]:
print("Running full validation scan on validation split...")
val_results = model.val()

print("\n" + "="*60)
print("YOLO11s-Seg Model Validation Summary:")
print("="*60)
print(f"Box Precision:  {val_results.results_dict['metrics/precision(B)']:.4f}")
print(f"Box Recall:     {val_results.results_dict['metrics/recall(B)']:.4f}")
print(f"Box mAP50:      {val_results.results_dict['metrics/mAP50(B)']:.4f}")
print(f"Box mAP50-95:   {val_results.results_dict['metrics/mAP50-95(B)']:.4f}")
print(f"Mask mAP50:     {val_results.results_dict['metrics/mAP50(M)']:.4f}")
print(f"Mask mAP50-95:  {val_results.results_dict['metrics/mAP50-95(M)']:.4f}")
print("="*60)

## 7. Interactive Test Cell: Upload Floorplan and Run Model
Run this cell to upload any floorplan image (PNG/JPG) from your computer and visualize predictions, counts, and segment overlays instantly.

In [ ]:
from google.colab import files
import cv2
import numpy as np
from IPython.display import Image, display

best_weights = os.path.join(run_dir, "weights", "best.pt")

if not os.path.exists(best_weights):
    print("ERROR: Trained best.pt weights not found. Make sure training completed successfully.")
else:
    print("Trained model loaded and ready. Please upload a floorplan image to test:")
    uploaded = files.upload()
    
    test_model = YOLO(best_weights)
    
    for fn in uploaded.keys():
        print(f"\nProcessing file: {fn}")
        # Execute Prediction
        res = test_model.predict(source=fn, imgsz=1024, conf=0.25)[0]
        
        door_cnt = 0
        win_cnt = 0
        
        if res.boxes is not None:
            for box in res.boxes:
                cls = int(box.cls[0].item())
                if cls == 0: door_cnt += 1
                elif cls == 1: win_cnt += 1
                
        print(f"\n" + "-"*30 + " Detections Summary " + "-"*30)
        print(f"Doors Identified:   {door_cnt}")
        print(f"Windows Identified: {win_cnt}")
        print(f"Total Takeoff Items: {door_cnt + win_cnt}")
        print("-"*80)
        
        # Draw predictions overlay
        plotted_img = res.plot()
        output_filename = f"predicted_{fn}"
        cv2.imwrite(output_filename, plotted_img)
        
        # Display original & prediction
        print(f"Predicted Segmentation Overlay ({output_filename}):")
        display(Image(filename=output_filename))